# 002-perfect-satellite-map

# Imports

In [ ]:
import json
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.data.drone_frames import DroneFrame, load_frames
from uavloc.data.satellite import GeoImage
from uavloc.features.factory import build_features, correlator_config
from uavloc.geo import LatLon
from uavloc.matching.benchmark import run_benchmark, summarise
from uavloc.matching.correlator import PositionCorrelator
from uavloc.track import REPO_ROOT, experiment_dir, log_run, set_seed

# Params

In [ ]:
EXP = "002-perfect-satellite-map"
METHOD = "segmentation-labels"            # "vgg11" | "segmentation" | "segmentation-labels"
MATCHING = "cosine"               # "cosine" (2025) | "weighted-ncc"
CLASS_WEIGHTS = (0.0, 1.0, 1.0, 0.5, 0.0, 0.5)   # background, building, track, woody, field, water; weighted-ncc only
NOTE = ""

NOISE_M = [5.0, 10.0, 15.0]
PRIMARY_NOISE_M = 15.0
N_DIRECTIONS = 8
SEED = 42
DEVICE = "mps"

# 2025 correlator geometry, unchanged for every method
SEARCH_SCALE = 1.7
FOV_W_DEG = 60.0
SAT_FEATURE_CROP_PCT = 0.03
DRONE_FEATURE_CROP_PCT = 0.0

VGG11_CHECKPOINT = "../archive/uav_localization_2025/feature_models/models_finetrain/vgg11_drop_margin08_adamw_model_20250521_034900.pth"
DRONE_INPUT_PX = 448

SEG_CHECKPOINT_SAT = "pocs/village-seg-baseline/model-round3-bn-satellite.pt"
SEG_CHECKPOINT_DRONE = "pocs/village-seg-baseline/model-round3-bn-drone.pt"
SEG_CHANNELS = "all"                # "all" | "no-background"
DRONE_SEG_GSD_M = 0.10
MATCH_GSD_M = 0.385

LABEL_LOCATION = "gonchary-2025"               # hand-labelled satellite map, segmentation-labels only
LABEL_DATE = "ortho"
LABEL_DATASET = "ua-ortho-annotated"

FRAMES_DATASET = "drone-2025-easy"
SEG_SPLITS = "pocs/village-seg-baseline/results-round2.json"
SAT_IMAGE = "../archive/uav_localization_2025/data/satelites/satellite_gonchary.jpg"
SAT_TOP_LEFT = (49.762965, 24.166472)
SAT_BOTTOM_RIGHT = (49.759291, 24.174046)

# Data

In [ ]:
set_seed(SEED)

In [ ]:
RUN_CONFIG = {
    "method": METHOD, "matching": MATCHING, "class_weights": CLASS_WEIGHTS,
    "noise_m": NOISE_M, "primary_noise_m": PRIMARY_NOISE_M, "n_directions": N_DIRECTIONS,
    "search_scale": SEARCH_SCALE, "fov_w_deg": FOV_W_DEG,
    "sat_feature_crop_pct": SAT_FEATURE_CROP_PCT, "drone_feature_crop_pct": DRONE_FEATURE_CROP_PCT,
    "vgg11_checkpoint": VGG11_CHECKPOINT, "drone_input_px": DRONE_INPUT_PX,
    "seg_checkpoint_sat": SEG_CHECKPOINT_SAT, "seg_checkpoint_drone": SEG_CHECKPOINT_DRONE,
    "seg_channels": SEG_CHANNELS, "drone_seg_gsd_m": DRONE_SEG_GSD_M, "match_gsd_m": MATCH_GSD_M,
    "label_location": LABEL_LOCATION, "label_date": LABEL_DATE, "label_dataset": LABEL_DATASET,
    "frames_dataset": FRAMES_DATASET, "sat_image": SAT_IMAGE,
}

In [ ]:
frames = load_frames(FRAMES_DATASET)
len(frames)

In [ ]:
satellite = GeoImage.load(REPO_ROOT / SAT_IMAGE, LatLon(*SAT_TOP_LEFT), LatLon(*SAT_BOTTOM_RIGHT))
satellite.mpp

In [ ]:
def frame_subsets(frames: list[DroneFrame], seg_splits_path: str) -> dict[str, set[str]]:
    splits = json.loads((REPO_ROOT / seg_splits_path).read_text())["splits"]
    seg_seen = set(splits["drone_train"] + splits["drone_dropped"])
    return {
        "all": {f.name for f in frames},
        "2025_train": {f.name for f in frames if "images_2_v3_renamed_small_easy" in f.picked_from},
        "2025_val": {f.name for f in frames if "images_v6_renamed_easy" in f.picked_from},
        "seg_heldout": {f.name for f in frames if f.name not in seg_seen},
    }

In [ ]:
subsets = frame_subsets(frames, SEG_SPLITS)
{name: len(members) for name, members in subsets.items()}

# Model

In [ ]:
features = build_features(RUN_CONFIG, satellite, DEVICE)
correlator = PositionCorrelator(features, correlator_config(RUN_CONFIG))

# Run

In [ ]:
records = pd.concat([run_benchmark(correlator, frames, noise_m, N_DIRECTIONS, SEED) for noise_m in NOISE_M], ignore_index=True)
records.groupby("noise_m")[["error_m", "reachable"]].mean()

In [ ]:
metrics = summarise(records, subsets, PRIMARY_NOISE_M)
{k: v for k, v in metrics.items() if "_at_" not in k}

# Plots

In [ ]:
fig_cdf, ax = plt.subplots(figsize=(6, 4))
for noise_m, group in records.groupby("noise_m"):
    errors = np.sort(group["error_m"].to_numpy())
    line, = ax.plot(errors, np.linspace(0, 1, len(errors)), label=f"{noise_m:g} m injected")
    ax.axvline(noise_m, color=line.get_color(), ls=":", lw=1)
ax.set_xlabel("position error after correction, m"); ax.set_ylabel("share of draws")
ax.set_title(f"{METHOD}, {MATCHING}"); ax.legend(); ax.grid(alpha=.3)
fig_cdf.tight_layout()

In [ ]:
primary = records[records["noise_m"] == PRIMARY_NOISE_M]
subset_means = {name: primary[primary["frame"].isin(members)]["error_m"].mean() for name, members in subsets.items()}
fig_subsets, ax = plt.subplots(figsize=(6, 3))
ax.bar(list(subset_means), list(subset_means.values()))
ax.axhline(PRIMARY_NOISE_M, color="k", ls=":", lw=1, label="no correction")
ax.set_ylabel(f"mean error @ {PRIMARY_NOISE_M:g} m, m"); ax.set_title(f"{METHOD}, {MATCHING}"); ax.legend()
fig_subsets.tight_layout()

In [ ]:
fig_reach, ax = plt.subplots(figsize=(6, 3.5))
for reachable, group in primary.groupby("reachable"):
    ax.hist(group["error_m"], bins=30, range=(0, 40), alpha=.6, label=f"truth in window: {reachable}")
ax.axvline(PRIMARY_NOISE_M, color="k", ls=":", lw=1)
ax.set_xlabel(f"error @ {PRIMARY_NOISE_M:g} m injected, m"); ax.set_ylabel("draws"); ax.set_title(f"{METHOD}, {MATCHING}"); ax.legend()
fig_reach.tight_layout()

In [ ]:
run = log_run(
    exp=EXP,
    config=RUN_CONFIG,
    metrics=metrics,
    note=NOTE,
    figures=[fig_cdf, fig_subsets, fig_reach],
    seed=SEED,
    device=DEVICE,
)

In [ ]:
artifacts = experiment_dir(EXP) / "artifacts"
artifacts.mkdir(exist_ok=True)
records.to_csv(artifacts / f"records-{run['run']}.csv", index=False)